Code for Breast Cancer data set for final project AAI 500 at USD
Authors:     
Angelo Walsh   



In [167]:
import pandas as pd


In [168]:
# import cancer data set
cancer_df = pd.read_csv('Breast_Cancer_METABRIC_Epic_Hospital.csv', sep=',')
cancer_df.head(1)

,Patient ID,Age at Diagnosis,Type of Breast Surgery,Cancer Type,Cancer Type Detailed,Cellularity,Chemotherapy,Pam50 + Claudin-low subtype,Cohort,ER status measured by IHC,...,Overall Survival Status,PR Status,Radio Therapy,Relapse Free Status (Months),Relapse Free Status,Sex,3-Gene classifier subtype,Tumor Size,Tumor Stage,Patient's Vital Status
0,MB-0000,75.65,Mastectomy,Breast Cancer,Breast Invasive Ductal Carcinoma,High,No,claudin-low,1.0,Positve,...,Living,Negative,Yes,138.65,Not Recurred,Female,ER+/HER2- High Prolif,22.0,2.0,Living


In [169]:
 # normalize column names. Set all column names to lowercase, replace spaces with underscores, and remove special characters
cancer_df.columns = (  
    cancer_df.columns
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", "_", regex=True)
    .str.replace(r"[^a-zA-Z0-9_]", "", regex=True)
    .str.replace(r"_+", "_", regex=True)
    .str.strip("_")
)
cancer_df.head(1)

,patient_id,age_at_diagnosis,type_of_breast_surgery,cancer_type,cancer_type_detailed,cellularity,chemotherapy,pam50_claudinlow_subtype,cohort,er_status_measured_by_ihc,...,overall_survival_status,pr_status,radio_therapy,relapse_free_status_months,relapse_free_status,sex,3gene_classifier_subtype,tumor_size,tumor_stage,patients_vital_status
0,MB-0000,75.65,Mastectomy,Breast Cancer,Breast Invasive Ductal Carcinoma,High,No,claudin-low,1.0,Positve,...,Living,Negative,Yes,138.65,Not Recurred,Female,ER+/HER2- High Prolif,22.0,2.0,Living


In [170]:
# Set age column to integer type
cancer_df['age_at_diagnosis'] = cancer_df['age_at_diagnosis'].astype(int)  # set age column to integer type
cancer_df.head(1)


,patient_id,age_at_diagnosis,type_of_breast_surgery,cancer_type,cancer_type_detailed,cellularity,chemotherapy,pam50_claudinlow_subtype,cohort,er_status_measured_by_ihc,...,overall_survival_status,pr_status,radio_therapy,relapse_free_status_months,relapse_free_status,sex,3gene_classifier_subtype,tumor_size,tumor_stage,patients_vital_status
0,MB-0000,75,Mastectomy,Breast Cancer,Breast Invasive Ductal Carcinoma,High,No,claudin-low,1.0,Positve,...,Living,Negative,Yes,138.65,Not Recurred,Female,ER+/HER2- High Prolif,22.0,2.0,Living


In [171]:
# Check for duplicate records
duplicate_rows = cancer_df.duplicated().sum()
duplicate_patient_ids = cancer_df["patient_id"].duplicated().sum()

print("Duplicate rows:", duplicate_rows)
print("Duplicate patient IDs:", duplicate_patient_ids)

Duplicate rows: 0
Duplicate patient IDs: 0


In [172]:
# Investigate replicated patient records
# Exclude patient_id when comparing clinical records
features = cancer_df.columns.drop("patient_id")

print("Total patients:", len(cancer_df))
print("Unique patient IDs:", cancer_df["patient_id"].nunique())
print("Unique clinical records:", cancer_df[features].drop_duplicates().shape[0])

Total patients: 15054
Unique patient IDs: 15054
Unique clinical records: 2509


Number of rows remaining is a division of 6 from the original data set. 15054 / 2509 = 6. This gives evidence that the data is repeated 6 times. 

In [173]:
# Show the distribution of group sizes for unique clinical records
group_sizes = (
    cancer_df
    .groupby(list(features), dropna=False)
    .size()
)

print(group_sizes.value_counts().sort_index())

6    2509
Name: count, dtype: int64


In [174]:
# Retain one observation from each group of replicated records
cancer_df_unique = cancer_df.drop_duplicates(
    subset=features,
    keep="first"
).copy()

print("Rows before:", len(cancer_df))
print("Rows after removing replicated records:", len(cancer_df_unique))

Rows before: 15054
Rows after removing replicated records: 2509


In [175]:
# Check for missing data in the unique clinical records
print("Original rows:", len(cancer_df_unique))
print("Rows after dropna:", len(cancer_df_unique.dropna()))
print("Rows lost:", len(cancer_df_unique) - len(cancer_df_unique.dropna()))
rows_lost_percentage = (len(cancer_df_unique) - len(cancer_df_unique.dropna())) / len(cancer_df_unique) * 100
print("Percentage of rows lost:", round(rows_lost_percentage, 4))

Original rows: 2509
Rows after dropna: 2481
Rows lost: 28
Percentage of rows lost: 1.116


In [176]:
# Summarize missing data
missing_summary = pd.DataFrame({
    "missing_count": cancer_df_unique.isna().sum(),
    "missing_percent": (cancer_df_unique.isna().mean() * 100).round(2)
})

missing_summary[missing_summary["missing_count"] > 0]

,missing_count,missing_percent
type_of_breast_surgery,1,0.04
cellularity,5,0.20
pam50_claudinlow_subtype,1,0.04
cohort,1,0.04
tumor_other_histologic_subtype,1,0.04
primary_tumor_laterality,3,0.12
lymph_nodes_examined_positive,1,0.04
mutation_count,10,0.40
nottingham_prognostic_index,1,0.04
pr_status,1,0.04


Columns with missing data have very low percentages, less than 0.4%. Missing percent for mutation count is 0.4, the next is cellularity at 0.2, the rest are even lower. 

In [177]:
# Drop missing data from the unique clinical records
cancer_df_unique_cleaned = cancer_df_unique.dropna()
print("Rows after dropping missing data:", len(cancer_df_unique_cleaned))

Rows after dropping missing data: 2481


In [178]:
# Export cleaned data set
cancer_df_unique_cleaned.to_csv("cancer_data_cleaned.csv", index=False)
print("Data exported to cancer_data_cleaned.csv")
print("Number of rows in cleaned data:", len(cancer_df_unique_cleaned))
print("Number of columns in cleaned data:", len(cancer_df_unique_cleaned.columns))

Data exported to cancer_data_cleaned.csv
Number of rows in cleaned data: 2481
Number of columns in cleaned data: 34
